In [2]:
!pip3 install tiktoken

In [3]:
import numpy as np
import pandas as pd
import tiktoken

In [4]:
tokenizer=tiktoken.get_encoding('gpt2')

In [10]:
with open('/content/the-verdict.txt','r',encoding='utf-8') as f:
  text=f.read()

In [11]:
tokens=tokenizer.encode(text,allowed_special={"<|endoftext|>"})

In [12]:
tokens

[40,
 367,
 2885,
 1464,
 1807,
 3619,
 402,
 271,
 10899,
 2138,
 257,
 7026,
 15632,
 438,
 2016,
 257,
 922,
 5891,
 1576,
 438,
 568,
 340,
 373,
 645,
 1049,
 5975,
 284,
 502,
 284,
 3285,
 326,
 11,
 287,
 262,
 6001,
 286,
 465,
 13476,
 11,
 339,
 550,
 5710,
 465,
 12036,
 11,
 6405,
 257,
 5527,
 27075,
 11,
 290,
 4920,
 2241,
 287,
 257,
 4489,
 64,
 319,
 262,
 34686,
 41976,
 13,
 357,
 10915,
 314,
 2138,
 1807,
 340,
 561,
 423,
 587,
 10598,
 393,
 28537,
 2014,
 198,
 198,
 1,
 464,
 6001,
 286,
 465,
 13476,
 1,
 438,
 5562,
 373,
 644,
 262,
 1466,
 1444,
 340,
 13,
 314,
 460,
 3285,
 9074,
 13,
 46606,
 536,
 5469,
 438,
 14363,
 938,
 4842,
 1650,
 353,
 438,
 2934,
 489,
 3255,
 465,
 48422,
 540,
 450,
 67,
 3299,
 13,
 366,
 5189,
 1781,
 340,
 338,
 1016,
 284,
 3758,
 262,
 1988,
 286,
 616,
 4286,
 705,
 1014,
 510,
 26,
 475,
 314,
 836,
 470,
 892,
 286,
 326,
 11,
 1770,
 13,
 8759,
 2763,
 438,
 1169,
 2994,
 284,
 943,
 17034,
 318,
 477,
 314,
 892,


In [7]:
len(tokens)

5145

In [13]:
import torch
from torch.utils.data import Dataset,DataLoader

In [14]:
class GPTDataset(Dataset):
  def __init__(self,text,tokenizer,context_length,stride):
    self.input_ids=[]
    self.output_ids=[]
    tokens=tokenizer.encode(text)
    for i in range(0,len(tokens)-context_length,stride):
      input=tokens[i:i+context_length]
      output=tokens[i+1:i+context_length+1]
      self.input_ids.append(torch.tensor(input,dtype=torch.long))
      self.output_ids.append(torch.tensor(output,dtype=torch.long))
  def __len__(self):
    return len(self.input_ids)
  def __getitem__(self, index):
    return self.input_ids[index],self.output_ids[index]


In [15]:
def customdataloader(text,tokenizer,context_length,stride,batch_size,drop_last=True,shuffle=True):
  dataset=GPTDataset(text,tokenizer,context_length,stride)
  dataloader=DataLoader(dataset,batch_size=batch_size,drop_last=drop_last,shuffle=shuffle)
  return dataloader


In [25]:
customdataloader(text,tokenizer=tokenizer,context_length=256,stride=128,batch_size=4)

In [26]:
data_iter=customdataloader(text,tokenizer=tokenizer,context_length=256,stride=128,batch_size=4)

In [34]:
import torch.nn as nn
vocab_size=tokenizer.n_vocab
embed_dim=256
token_embeddings_layer=nn.Embedding(vocab_size,embed_dim)
positional_embeddings_layer=nn.Embedding(256,embed_dim)
for batch_x,batch_y in data_iter :
  batch_x_token=token_embeddings_layer(batch_x)
  batch_x_position=positional_embeddings_layer(torch.arange(256))
  batch_y_token=token_embeddings_layer(batch_y)
  batch_y_position=positional_embeddings_layer(torch.arange(256))
  batch_x=batch_x_position+batch_x_token
  batch_y=batch_y_position+batch_y_token


